In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS lshc_lakehouse.cpcb_aqi;

In [0]:
%sql
SELECT 
    bronze_ingested_at,
    source_file,
    title,
    total,
    count,
    records
FROM lshc_lakehouse.cpcb_aqi.bronze_cpcb_telemetry
ORDER BY bronze_ingested_at DESC
LIMIT 5;

In [0]:
%sql
DESCRIBE EXTENDED lshc_lakehouse.`01_bronze`.bronze_cpcb_telemetry;

In [0]:
%sql
-- dbutils.fs.ls() is blocked on UC managed storage paths (__unitystorage).
-- Use DESCRIBE DETAIL to inspect table storage metadata instead.
DESCRIBE DETAIL lshc_lakehouse.cpcb_aqi.bronze_cpcb_telemetry;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS lshc_lakehouse.cpcb_aqi_custom
MANAGED LOCATION 'abfss://aqi-data@lshc.dfs.core.windows.net/delta_tables/';

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS lshc_lakehouse
MANAGED LOCATION 'abfss://aqi-data@lshc.dfs.core.windows.net/delta_tables/';

In [0]:
%sql
DROP SCHEMA lshc_lakehouse.01_bronze CASCADE;

In [0]:
%sql
CREATE SCHEMA lshc_lakehouse.01_bronze
  MANAGED LOCATION 'abfss://aqi-data@lshc.dfs.core.windows.net/lshc-lakehouse/01_bronze';

In [0]:
%sql DESCRIBE SCHEMA EXTENDED aqi_data.bronze

In [0]:
# 1. Inspect delta_tables (Your Delta Table Parquet + Logs)
display(dbutils.fs.ls("abfss://aqi-data@lshc.dfs.core.windows.net/delta_tables/"))

# 2. Inspect lshc-lakehouse (DLT Pipeline Logs & Checkpoints)
display(dbutils.fs.ls("abfss://aqi-data@lshc.dfs.core.windows.net/lshc-lakehouse/"))

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS lshc_lakehouse.02_silver
  MANAGED LOCATION 'abfss://aqi-data@lshc.dfs.core.windows.net/delta_tables/02_silver';

In [0]:
%sql
DROP catalog lshc_lakehouse cascade

In [0]:
%sql
DROP SCHEMA IF EXISTS aqi_data.gold CASCADE;

In [0]:
%sql
CREATE SCHEMA aqi_data.bronze 
MANAGED LOCATION 'abfss://aqi-data@lshc.dfs.core.windows.net/delta_tables/01_bronze/';

In [0]:
%sql
-- Creates the silver schema and strictly routes all physical data and Delta logs
-- to your custom Azure ADLS folder, bypassing Unity Catalog's default storage.
CREATE SCHEMA IF NOT EXISTS aqi_data.silver
MANAGED LOCATION 'abfss://aqi-data@lshc.dfs.core.windows.net/delta_tables/02_silver/';

In [0]:
%sql
-- Create the gold schema and route physical data to the 03_gold folder
CREATE SCHEMA IF NOT EXISTS aqi_data.gold
MANAGED LOCATION 'abfss://aqi-data@lshc.dfs.core.windows.net/delta_tables/03_gold/';